In [21]:
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.variant_scoring import (
    VariantScoringModel, Variant, Interval,
    CenterMaskScorer, OutputType, AggregationType,
    get_recommended_scorers,
)
from alphagenome_pytorch.variant_scoring.scorers.gene_mask import GeneMaskMode


In [ ]:

# Load model (track_means are bundled in weights from convert_weights.py)
model = AlphaGenome.from_pretrained('data/model_related/model_all_folds.safetensors', device='cuda')


In [ ]:
# Create scoring wrapper
scoring_model = VariantScoringModel(
    model,
    fasta_path='data/model_related/hg38.fa',
    gtf_path='data/annotations/gencode.v46.annotation.parquet',
    polya_path='data/annotations/gencode.v46.polyAs.linked.parquet',  # Optional, for PolyadenylationScorer
    default_organism='human',
)
scoring_model.load_all_metadata('track_metadata.parquet')

In [4]:
# load scorere that alphagenome used for eQTL scoring in the paper
from alphagenome_pytorch.variant_scoring import GeneMaskLFCScorer

rna_scorer = GeneMaskLFCScorer(
    requested_output=OutputType.RNA_SEQ,
    mask_mode=GeneMaskMode.EXONS,  # EXONS or BODY
    resolution=1,  # Default 128bp for gene-level
)

In [5]:
def score_variant_pytorch(variant_id):
    variant = Variant.from_str(variant_id, format="gtex")

    interval = Interval.centered_on(
        variant.chromosome,
        variant.position
    )

    scores = scoring_model.score_variant(
        interval=interval,
        variant=variant,
        scorers=[rna_scorer],
        to_cpu=True,
    )

    return scoring_model.tidy_scores(scores)

In [6]:
urls = [
    "https://storage.googleapis.com/alphagenome/evals/eqtl_variant_borzoi_coefficient_human_predictions.feather",
    "https://storage.googleapis.com/alphagenome/evals/eqtl_variant_catalogue_causality_gene_balanced_human_predictions.feather",
]

dfs = [pd.read_feather(url) for url in urls]

df_coefficient = dfs[0]
df_coefficient = df_coefficient.dropna(subset=["target"])
df_zeroshot = dfs[1]
df_zeroshot = df_zeroshot.dropna(subset=["target"])

In [15]:

import gc
import torch
from tqdm import tqdm

unique_variants = df_coefficient["variant_id"].unique()
unique_variants = unique_variants[:len(unique_variants) // 10]
variant_scores_dic = {}

for variant in tqdm(unique_variants):
    try:
        with torch.inference_mode():
            result = score_variant_pytorch(variant)
        variant_scores_dic[variant] = result
        
    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at {variant}")
        torch.cuda.empty_cache()
        gc.collect()
        variant_scores_dic[variant] = None
        continue

    except ValueError as e:
        print(f"Skipping {variant}: {e}")
        variant_scores_dic[variant] = None

    except Exception as e:
        print(f"Failed {variant}: {type(e).__name__}: {e}")
        variant_scores_dic[variant] = None



  2%|▏         | 15/620 [00:19<12:41,  1.26s/it]

Skipping chr3_29280576_T_C_b38: Reference allele mismatch at chr3:29280576. Expected 'T', found 'C' in sequence


  3%|▎         | 18/620 [00:21<10:22,  1.03s/it]

Skipping chr3_44590720_T_G_b38: Reference allele mismatch at chr3:44590720. Expected 'T', found 'G' in sequence


  3%|▎         | 21/620 [00:24<09:42,  1.03it/s]

...

Skipping chr19_38753481_G_A_b38: Reference allele mismatch at chr19:38753481. Expected 'G', found 'A' in sequence


 99%|█████████▉| 616/620 [30:53<00:42, 10.73s/it]

Skipping chr19_43205663_C_T_b38: Reference allele mismatch at chr19:43205663. Expected 'C', found 'T' in sequence


100%|██████████| 620/620 [31:14<00:00,  3.02s/it]

Skipping chr19_44087207_A_G_b38: Reference allele mismatch at chr19:44087207. Expected 'A', found 'G' in sequence
Skipping chr19_44718495_C_T_b38: Reference allele mismatch at chr19:44718495. Expected 'C', found 'T' in sequence


In [18]:
variant_scores_dic = {
    k: v for k, v in variant_scores_dic.items() if v is not None
}

In [22]:
df_coefficient_results = []

for row_nr in range(len(df_coefficient)):
    row = df_coefficient.iloc[row_nr]
    variant = row.variant_id
    
    if variant not in variant_scores_dic:
        continue
    
    variant_scores = variant_scores_dic[variant]
    
    mask = (
        (variant_scores["gene_id"] == row.gene_id.split('.')[0]) &
        (variant_scores["gtex_tissue"] == row.tissue)
    )

    filtered = variant_scores.loc[mask, "raw_score"]

    new_prediction = filtered.iloc[0] if not filtered.empty else np.nan
    # add prediction from model to the row and then collect row in a new dataframe for all variants
    new_row = row.copy()
    new_row['prediction_new'] = new_prediction

    df_coefficient_results.append(new_row)


In [23]:
# output dataframe to file
import pandas as pd

df_results_coefficient = pd.DataFrame(df_coefficient_results)
df_results_coefficient.to_feather("results/pytorch/coefficients_part1.feather")

In [24]:
df_results_coefficient = pd.read_feather("results/pytorch/coefficients_part1.feather")

In [27]:
df_results_coefficient

,variant_id,gene_id,tissue,prediction,target,variant_scorer,output_type,metric_calculator,metric_name,prediction_new
0,chr3_4491276_T_C_b38,ENSG00000231249.3,Adipose_Subcutaneous,0.101731,0.279811,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,0.093171
1,chr3_8733693_G_T_b38,ENSG00000182533.7,Adipose_Subcutaneous,0.014747,0.567912,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.011369
2,chr3_8963418_C_T_b38,ENSG00000070950.10,Adipose_Subcutaneous,-0.095976,-0.684296,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.136670
3,chr3_9867065_G_C_b38,ENSG00000187288.12,Adipose_Subcutaneous,-0.000609,-0.130677,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.007301
4,chr3_9903375_C_T_b38,ENSG00000163701.19,Adipose_Subcutaneous,-0.035684,-0.660048,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.019136
...,...,...,...,...,...,...,...,...,...,...
318768,chr19_36114973_G_T_b38,ENSG00000105254.12,Whole_Blood,0.052913,0.453086,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,0.058266
318780,chr19_40750305_G_T_b38,ENSG00000188493.15,Whole_Blood,0.085684,1.149064,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,0.092220
318781,chr19_40909182_G_A_b38,ENSG00000256612.9,Whole_Blood,-0.047596,1.018026,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.125596
318785,chr19_41586462_A_T_b38,ENSG00000007129.18,Whole_Blood,-0.670920,-0.802152,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataCorrelations,None,-0.590636


## Zeroshot

In [ ]:

import gc
import torch
from tqdm import tqdm

unique_variants = df_zeroshot["variant_id"].unique()
unique_variants = unique_variants[:len(unique_variants) // 10]
variant_scores_dic = {}

for variant in tqdm(unique_variants):
    try:
        with torch.inference_mode():
            result = score_variant_pytorch(variant)
        variant_scores_dic[variant] = result
        
    except torch.cuda.OutOfMemoryError:
        print(f"CUDA OOM at {variant}")
        torch.cuda.empty_cache()
        gc.collect()
        variant_scores_dic[variant] = None
        continue

    except ValueError as e:
        print(f"Skipping {variant}: {e}")
        variant_scores_dic[variant] = None

    except Exception as e:
        print(f"Failed {variant}: {type(e).__name__}: {e}")
        variant_scores_dic[variant] = None



100%|██████████| 5/5 [01:07<00:00, 13.59s/it]


In [38]:
df_zeroshot_results = []

for row_nr in range(len(df_zeroshot)):
    row = df_zeroshot.iloc[row_nr]
    variant = row.variant_id
    
    if variant not in variant_scores_dic:
        continue
    
    variant_scores = variant_scores_dic[variant]
    
    mask = (
        (variant_scores["gene_id"] == row.gene_id.split('.')[0]) &
        (variant_scores["gtex_tissue"] == row.tissue)
    )

    filtered = variant_scores.loc[mask, "raw_score"]

    new_prediction = filtered.iloc[0] if not filtered.empty else np.nan
    # add prediction from model to the row and then collect row in a new dataframe for all variants
    new_row = row.copy()
    new_row['prediction_new'] = new_prediction

    df_zeroshot_results.append(new_row)


In [39]:

df_results = pd.DataFrame(df_zeroshot_results)
df_results.to_feather("results/pytorch/zeroshot_test.feather")

In [ ]:
df_results = pd.read_feather("results/pytorch/zeroshot_test.feather")

In [40]:
df_results.head()

,variant_id,gene_id,tissue,prediction,target,variant_scorer,output_type,metric_calculator,metric_name,prediction_new
0,chr6_35141318_A_G,ENSG00000124678,Adipose_Subcutaneous,0.015841,0.0,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataAUROC,None,0.042902
16,chr16_84924808_A_G,ENSG00000279622,Adipose_Subcutaneous,0.237943,1.0,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataAUROC,None,0.188671
17,chr18_46104399_G_C,ENSG00000152240,Adipose_Subcutaneous,0.249200,1.0,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataAUROC,None,0.299480
18,chr19_49691588_C_A,ENSG00000169169,Adipose_Subcutaneous,0.177043,1.0,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataAUROC,None,0.140100
19,chr19_53365706_G_C,ENSG00000203326,Adipose_Subcutaneous,0.030469,1.0,gene_exons_contained_diff_mean;OutputType.RNA_...,RNA_SEQ,TissueAggregatedAnnDataAUROC,None,-0.058027
